In [1]:
#from src.kgb import rewrite_conclusion_as_binary_qa

In [2]:
import pandas as pd # Optional, but great for viewing the final list
df=pd.read_csv('data/conclusion_result.csv')
# 1. Define your list of scientific statements
my_statements = [
    "Plasma cfChIP-seq captures hepatocyte disease-specific gene activity in AIH patients.",
    "Daily consumption of green tea significantly reduces the risk of cardiovascular disease in adults over 50.",
    "The experimental vaccine did not show a statistically significant improvement in neutralizing the virus compared to the placebo."
]


In [3]:
df[df['conclusion'].isna()][['doi','abstract']].to_csv('data/t.csv')

In [ ]:


# 2. Create an empty list to store your successfully extracted data
qa_results = []

# 3. Loop through each statement
for i, statement in enumerate(my_statements):
    print(f"Processing statement {i+1}/{len(my_statements)}...")
    
    # Run the LLM task
    run_result = rewrite_conclusion_as_binary_qa.run(kbench.llm, conclusion=statement)
    
    try:
        # First, ensure the kbench task actually succeeded
        if run_result.status.name == 'SUCCESS':
            
            # Extract the Pydantic object using the path we found earlier
            llm_output = run_result.chat.history[1].content
            
            # Store the data in a standard Python dictionary
            qa_results.append({
                "Original Statement": statement,
                "Question": llm_output.question,
                "Answer": llm_output.answer
            })
            print(f"  ✓ Success: Answered '{llm_output.answer}'")
            
        else:
            print(f"  ✗ Task failed or was rejected. Status: {run_result.status}")
            
    except (IndexError, AttributeError) as e:
        # This catches edge cases where the chat history is empty or formatted weirdly
        print(f"  ✗ Failed to parse output structure. Error: {e}")

# 4. View the results (using Pandas makes it look like a nice table)
print("\n--- Final Extracted Data ---")
df_results = pd.DataFrame(qa_results)
print(df_results)